# AI Lab - Practical Question No. 28
## A* Search Algorithm on a Graph (Informed Search & Heuristics)

- **Syllabus Area**: Informed Search and Heuristics
- **Difficulty**: 🟠 Advanced
- **Practical Type**: Implementation
- **Core AI Concept**: A* Search, Optimal and Complete Informed Search ($f(n) = g(n) + h(n)$)
- **Python Concepts Tested**: Priority queues via `heapq`, dynamic dictionary updating, lazy deletion
- **Expected Algorithm / Technique**: A* Graph Search
- **Recommended Python Libraries**: `heapq`, `dataclasses`, `typing`, `networkx`, `matplotlib`
- **Complexity Expectation**: Time: $\mathcal{O}(E \log V)$, Space: $\mathcal{O}(V)$
- **Constraints / Conditions**: Heuristics are admissible ($0 \le h(n) \le h^*(n)$)
- **Input Requirements**: Weighted graph with edge weights, Heuristic dictionary
- **Expected Output**: Optimal path list and integer total cost
- **Sample Input**: Start: `S`, Goal: `G`
- **Sample Output**: `Optimal Path: S -> C -> G, Total Cost: 9`
- **Optional Challenge**: Print the frontier contents at each step
- **University Practical Importance**: Very High

---

### Problem Statement
Write a Python program to implement the **A* Search algorithm** on a weighted graph. The program must evaluate nodes using the function:

$$f(n) = g(n) + h(n)$$

and find the optimal path from source to destination.

#### Task Requirements:
1. **Maintain `g_cost`** for each visited node.
2. **Use a priority queue** sorted by `f_cost` via Python's `heapq` module.
3. **Allow re-evaluation** of nodes if a cheaper `g_cost` is found.
4. **Output optimal path and total cost**.

## 1. Theoretical Foundations: Components of A* Search

A* Search evaluates candidate frontier states by combining the backward cost incurred so far with the forward estimated cost to the target:

$$f(n) = g(n) + h(n)$$

### Mathematical Definitions:
1. **$g(n)$ (Path Cost)**: The exact accumulated cost to reach node $n$ from the start node $S$.
2. **$h(n)$ (Heuristic Function)**: An estimated cost to travel from node $n$ to the nearest goal state $G$.
3. **$f(n)$ (Evaluation Function)**: The estimated total cost of the cheapest solution path passing through node $n$.

### Conditions for Optimality:
- **Admissibility ($0 \le h(n) \le h^*(n)$)**: The heuristic never overestimates the true remaining distance to the goal. Admissibility guarantees optimality for **Tree Search**.
- **Consistency / Monotonicity ($h(u) \le c(u, v) + h(v)$)**: Satisfies the triangle inequality along every edge. Consistency guarantees that $f(n)$ values never decrease along any search path, ensuring that when a node is selected for expansion, its $g(n)$ cost is already optimal (eliminating the need to reopen closed nodes in **Graph Search**).
- **Reopening Closed Nodes**: If an admissible heuristic is non-monotonic (inconsistent), graph search remains optimal if closed nodes can be reopened upon finding a lower $g(n)$.

In [ ]:
import sys
from pathlib import Path

# Ensure repository root is on sys.path for local package imports
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.a_star_search import (
    DEFAULT_SAMPLE_GRAPH,
    DEFAULT_SAMPLE_HEURISTICS,
    ROMANIA_MAP_GRAPH,
    ROMANIA_SLD_HEURISTICS,
    REOPENING_DEMO_GRAPH,
    REOPENING_INCONSISTENT_HEURISTICS,
    DIAMOND_GRAPH,
    DIAMOND_HEURISTICS,
    AStarSearch,
    a_star_search,
    check_admissibility,
    check_consistency,
    compare_astar_vs_ucs_and_greedy,
    get_a_star_viva_reference,
    visualize_a_star,
)

print("A* Search module and benchmarks loaded successfully.")

## 2. Canonical Syllabus Sample Execution

Let us run the canonical syllabus instance:
- **Graph**:
  - `S -> [('A', 3), ('B', 1), ('C', 4)]`
  - `B -> [('A', 1)]` (discovering cheaper path to A: $1 + 1 = 2 < 3$)
  - `A -> [('G', 8)]` (path $S \to B \to A \to G$ has cost $2 + 8 = 10$)
  - `C -> [('G', 5)]` (path $S \to C \to G$ has cost $4 + 5 = 9$)
- **Admissible Heuristics**: `{'S': 7, 'A': 6, 'B': 7, 'C': 4, 'G': 0}`
- **Start**: `'S'`, **Goal**: `'G'`

In [ ]:
result = a_star_search(
    DEFAULT_SAMPLE_GRAPH,
    DEFAULT_SAMPLE_HEURISTICS,
    start="S",
    goal="G",
)

print("Canonical Output:")
print(result.format_path())
print(f"\nTotal Nodes Expanded ({len(result.nodes_expanded)}): {' -> '.join(result.nodes_expanded)}")
print(f"Cheaper g_cost Re-evaluations: {result.re_evaluations_count}")
print(f"Execution Time: {result.execution_time_ms:.3f} ms")
print("\nStep-by-Step Priority Queue Trace Table:")
print(result.format_step_table())

## 3. Optional Challenge: Tracking Frontier Contents at Each Step

The university practical includes an **Optional Challenge**: *Print the frontier contents at each step*.

This illustrates:
1. Insertion of newly generated successors into the min-heap ordered by $f(n)$.
2. Dynamic updating of $g(n)$ when a cheaper path is found.
3. Lazy deletion of superseded entries ($g(n) > \text{best\_g}(n)$) in $\mathcal{O}(1)$ time.

In [ ]:
print(result.format_frontier_at_steps())

## 4. Re-Evaluation & Reopening Demonstration

When a heuristic is **admissible but NOT consistent** (non-monotonic), an optimal path may pass through a node that was previously expanded with a suboptimal cost.

In this demonstration graph:
- $S \to A$ ($c=5, h=0 \implies f=5$)
- $S \to B$ ($c=1, h=6 \implies f=7$)
- $B \to A$ ($c=2$)
- $A \to G$ ($c=4$)

Because $f(A) = 5 < f(B) = 7$, node $A$ is expanded **first** with $g(A) = 5$. Later, when $B$ is expanded, a cheaper route $S \to B \to A$ with $g(A) = 3$ is discovered! We demonstrate how **reopening** closed nodes guarantees finding the true optimal path.

In [ ]:
# 1. With Reopening Enabled:
res_reopen = a_star_search(
    REOPENING_DEMO_GRAPH,
    REOPENING_INCONSISTENT_HEURISTICS,
    start="S",
    goal="G",
    allow_reopen=True,
)
print("[1] With Reopening Enabled (allow_reopen=True):")
print(f"    {res_reopen.format_path()} (Guaranteed Optimal!)")
print(f"    Closed Nodes Reopened: {res_reopen.nodes_reopened_count}")
print(f"    Expansion Sequence: {' -> '.join(res_reopen.nodes_expanded)}")

# 2. With Reopening Disabled:
res_no_reopen = a_star_search(
    REOPENING_DEMO_GRAPH,
    REOPENING_INCONSISTENT_HEURISTICS,
    start="S",
    goal="G",
    allow_reopen=False,
)
print("\n[2] With Reopening Disabled (allow_reopen=False):")
print(f"    {res_no_reopen.format_path()} (Suboptimal: missed cheaper path!)")
print(f"    Expansion Sequence: {' -> '.join(res_no_reopen.nodes_expanded)}")

## 5. Classical AIMA Benchmark: Romania Map (Arad to Bucharest)

From Russell & Norvig's *Artificial Intelligence: A Modern Approach*, navigating from **Arad** to **Bucharest** using straight-line distances ($h_{\text{SLD}}$) to Bucharest:
- **Optimal Path**: `Arad -> Sibiu -> Rimnicu Vilcea -> Pitesti -> Bucharest`
- **Optimal Distance**: **418 km**

In [ ]:
romania_res = a_star_search(
    ROMANIA_MAP_GRAPH,
    ROMANIA_SLD_HEURISTICS,
    start="Arad",
    goal="Bucharest",
)
print(f"Result: {romania_res.format_path()}")
print(f"Nodes Expanded ({len(romania_res.nodes_expanded)}): {' -> '.join(romania_res.nodes_expanded)}")
print(f"Execution Time: {romania_res.execution_time_ms:.3f} ms")

## 6. Comparative Analysis: A* vs. UCS vs. Greedy Best-First Search

We compare the three core best-first graph search algorithms on the same graph:
1. **Uniform Cost Search (UCS)**: $f(n) = g(n)$ (Optimal, but blind).
2. **Greedy Best-First Search (GBFS)**: $f(n) = h(n)$ (Informed, but suboptimal).
3. **A* Search**: $f(n) = g(n) + h(n)$ (Informed and optimal).

In [ ]:
comp = compare_astar_vs_ucs_and_greedy(DEFAULT_SAMPLE_GRAPH, DEFAULT_SAMPLE_HEURISTICS, start="S", goal="G")

print(f"{'Algorithm':<28} | {'Evaluation Fn':<18} | {'Path':<12} | {'Cost':<5} | {'Expansions'}")
print("-" * 75)
for key, name in [("a_star", "A* Search"), ("uniform_cost_search", "Uniform Cost Search"), ("greedy_best_first_search", "Greedy Best-First")]:
    data = comp[key]
    path_str = "->".join(data["path"]) if data["path"] else "None"
    print(f"{name:<28} | {data['evaluation']:<18} | {path_str:<12} | {data['total_cost']:<5} | {data['expansion_count']} nodes")

## 7. Heuristic Verification: Admissibility and Consistency

We verify whether our heuristic satisfies:
1. **Admissibility**: $h(n) \le h^*(n)$ for all nodes $n$.
2. **Consistency (Monotonicity)**: $h(u) \le c(u, v) + h(v)$ for every edge $(u, v)$.

In [ ]:
adm_report = check_admissibility(DEFAULT_SAMPLE_GRAPH, DEFAULT_SAMPLE_HEURISTICS, goal="G")
cons_report = check_consistency(DEFAULT_SAMPLE_GRAPH, DEFAULT_SAMPLE_HEURISTICS)

print(f"Is Admissible?  : {adm_report['is_admissible']}")
print(f"Is Consistent?  : {cons_report['is_consistent']}")
print("\nNode Comparison Table (h vs h*):")
for node, detail in adm_report["comparisons"].items():
    print(f"  Node {node:<2}: h={detail['h']:<3} <= h*={detail['h_star']:<3} (Valid: {detail['is_valid']})")

## 8. Graphical Visualization

We visualize the graph topology with edge weights and node heuristics, highlighting the optimal path $S \to C \to G$ in red.

In [ ]:
img_path = visualize_a_star(
    result,
    output_path="astar_graph.png",
    title="A* Search on Weighted Graph: S -> C -> G (Cost: 9)",
)
if img_path:
    print(f"Visualization saved to: {img_path}")
    from IPython.display import Image, display
    display(Image(filename=img_path))
else:
    print("Visualization libraries (networkx/matplotlib) not installed.")

## 9. University Viva Voce Examination Reference Guide

Key theoretical questions, definitions, and model answers for viva voce preparation.

In [ ]:
viva = get_a_star_viva_reference()
print(f"=== {viva['title']} ===\n")
print(f"Core Formula: {viva['core_formula']['f_n']}")
for comp, desc in viva["core_formula"]["components"].items():
    print(f"  • {comp:<5} : {desc}")

print("\n" + "=" * 75)
for idx, qa in enumerate(viva["questions_and_answers"], 1):
    print(f"\n[Q{idx}] {qa['question']}")
    print(f"Answer:\n{qa['answer']}")